# DocuMind AI — train the clause classifier on a free GPU

Runtime → Change runtime type → **T4 GPU**, then run the cells in order.
Trains on all 60,000 LEDGAR clauses and scores on the full 10,000-clause test set (the same test set as the TF-IDF baseline).

In [ ]:
# Safe to run again: always starts from /content, clones once, then pulls the latest code.
%cd /content
![ -d documind_ai ] || git clone https://github.com/Hammad-cue/documind_ai.git
%cd /content/documind_ai
!git pull -q
!pip -q install -U "transformers>=5,<6" "datasets>=3" "accelerate>=1" scikit-learn

In [ ]:
%cd /content/documind_ai
# Small, fast model (same family the app already uses). ~15 min on a T4.
!python ml/train_clause_classifier.py --model sentence-transformers/all-MiniLM-L6-v2 --epochs 5 --batch-size 64 --max-length 256 --lr 1e-4

In [ ]:
%cd /content/documind_ai
# Optional, stronger and slower (~45 min on a T4): a larger encoder.
# !python ml/train_clause_classifier.py --model microsoft/deberta-v3-base --epochs 3 --batch-size 32 --max-length 256 --lr 3e-5 --out ml/models/clause-classifier-deberta

In [ ]:
%cd /content/documind_ai
# Download every model trained in this session (MiniLM and/or DeBERTa) with its metrics.
# Put them in your project as: ml/models/<folder>/  and  ml/runs/<timestamp>/metrics.json
import glob
models = [d.rsplit('/', 1)[0] for d in glob.glob('ml/models/*/config.json')]
assert models, 'No trained model here - the runtime was probably reset. Run cell 1 and a training cell again.'
print('Zipping:', models)
!zip -qr clause_model.zip ml/models ml/runs
from google.colab import files
files.download('clause_model.zip')